In [19]:
#单环境
import json
import networkx as nx
import numpy as np
import random
import os
from collections import Counter, defaultdict
from pathlib import Path

class ToolChainGenerator:
    def __init__(self, graph_path, steps = None):
        self.graph_path = graph_path
        self.file_name = os.path.basename(graph_path)
        
        with open(graph_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        self.G = nx.node_link_graph(data, directed=True)
        self.nodes = list(self.G.nodes())
        self.num_nodes = self.G.number_of_nodes()
        self.steps = self.num_nodes + 3 if steps is None else steps

    def _softmax(self, weights, temperature=1.0):
        weights = np.array(weights)
        e_x = np.exp((weights - np.max(weights)) / temperature)
        return e_x / e_x.sum()

    def generate(self, num_walks_per_node=3, temperature=1.0, 
                 max_repeats=3, base_stop_prob=0.05, stop_steps = 5):
        """
        :param max_repeats: 单个节点允许的最大重复次数 (硬刹车)
        :param base_stop_prob: 每一步的基础停止概率 (软刹车)
        """
        chains = []
        isolates = list(nx.isolates(self.G))
        for node in isolates:
            chains.append([node])

        active_nodes = [n for n in self.nodes if n not in isolates]
        
        # 绝对上限仍然是节点总数 (防止极端情况)
        abs_max_len = self.steps

        for start_node in active_nodes:
            for _ in range(num_walks_per_node):
                chain = [start_node]
                curr = start_node
                
                # 使用 Counter 实时跟踪当前链的节点重复情况
                node_counts = defaultdict(int)
                node_counts[curr] += 1
                
                while len(chain) < abs_max_len:
                    # 1. 获取所有邻居
                    neighbors = list(self.G.successors(curr))
                    if not neighbors:
                        break # 死胡同，自然停止

                    # 2. 【硬刹车】过滤掉重复次数超标的节点
                    valid_neighbors = [n for n in neighbors if node_counts[n] < max_repeats]
                    
                    if not valid_neighbors:
                        break # 周围的节点都访问太多次了，强制停止

                    # 3. 【软刹车】概率性终止 (Probabilistic Termination)
                    # 动态计算停止概率：基础概率 + (当前长度 * 0.02)
                    # 例子：长度为 5 时，停止概率 = 0.05 + 0.10 = 15%
                    # 长度为 10 时，停止概率 = 0.05 + 0.20 = 25%
                    current_stop_prob = base_stop_prob + (len(chain) * 0.01)
                    
                    # 只有当链长度至少为 5 时才允许概率停止（避免大量单步链）
                    if len(chain) >= stop_steps and random.random() < current_stop_prob:
                        break 

                    # 4. 准备权重并采样
                    weights = [self.G[curr][n].get('weight', 1.0) for n in valid_neighbors]
                    probs = self._softmax(weights, temperature)
                    
                    next_node = np.random.choice(valid_neighbors, p=probs)
                    
                    chain.append(next_node)
                    node_counts[next_node] += 1
                    curr = next_node
                
                chains.append(chain)
                
        return chains

    def save_and_report(self, chains, output_dir):
        # 去重
        unique_chains_tuple = set(tuple(c) for c in chains)
        unique_chains = [list(c) for c in unique_chains_tuple]
        
        total = len(unique_chains)
        if total == 0: return

        lengths = [len(c) for c in unique_chains]
        
        # 简单的文本直方图，帮你快速看分布
        hist_bins = [0, 5, 10, 20, 50, 100]
        hist_counts, _ = np.histogram(lengths, bins=hist_bins)
        
        print(f"📊 Report: {self.file_name}")
        print(f"  • Chains: {total} | Avg Len: {np.mean(lengths):.2f} | Max Len: {max(lengths)}")
        print(f"  • Distro: <5: {hist_counts[0]}, 5-10: {hist_counts[1]}, 10-20: {hist_counts[2]}, >20: {sum(hist_counts[3:])}")
        
        # 只有当所有链都触顶时才发出警告
        if min(lengths) == self.num_nodes:
            print(f"  ⚠️ WARNING: All chains hit max length! Increase stop_prob.")

        os.makedirs(output_dir, exist_ok=True)
        output_path = os.path.join(output_dir, f"chains_{self.file_name}")
        
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump({"meta": {"avg_len": np.mean(lengths)}, "chains": unique_chains}, f)

# --- 批量运行 ---
def process_all(input_dir, output_dir):
    files = list(Path(input_dir).glob("*.json"))
    for f in files:
        base_stop_prob = 0.1
        stop_steps = 5
        if 'math' in str(f) or 'memory' in str(f):
            base_stop_prob += 0.3
            stop_steps = 2
        print(f, stop_steps, base_stop_prob)
        gen = ToolChainGenerator(str(f))
        # 设置 max_repeats=3 防止死循环
        # 设置 base_stop_prob=0.1 (10% 基础概率停止)
        chains = gen.generate(
            max_repeats=3, 
            base_stop_prob=base_stop_prob, 
            temperature=1.2,
            stop_steps=stop_steps
        )
        gen.save_and_report(chains, output_dir)

process_all("./tool_env_graphs", "./file_output_chains")

tool_env_graphs/AutomotiveServiceRepairSystem_tool_graph.json 5 0.1
📊 Report: AutomotiveServiceRepairSystem_tool_graph.json
  • Chains: 51 | Avg Len: 8.84 | Max Len: 20
  • Distro: <5: 0, 5-10: 33, 10-20: 17, >20: 1
tool_env_graphs/CRMSystem_tool_graph.json 5 0.1
📊 Report: CRMSystem_tool_graph.json
  • Chains: 63 | Avg Len: 8.56 | Max Len: 18
  • Distro: <5: 0, 5-10: 45, 10-20: 18, >20: 0
tool_env_graphs/ChatApplicationBackend_tool_graph.json 5 0.1
📊 Report: ChatApplicationBackend_tool_graph.json
  • Chains: 60 | Avg Len: 9.58 | Max Len: 18
  • Distro: <5: 0, 5-10: 33, 10-20: 27, >20: 0
tool_env_graphs/CorporateFinancialReportingSystem_tool_graph.json 5 0.1
📊 Report: CorporateFinancialReportingSystem_tool_graph.json
  • Chains: 75 | Avg Len: 9.48 | Max Len: 22
  • Distro: <5: 0, 5-10: 40, 10-20: 34, >20: 1
tool_env_graphs/DataBackupRecoverySystem_tool_graph.json 5 0.1
📊 Report: DataBackupRecoverySystem_tool_graph.json
  • Chains: 51 | Avg Len: 8.29 | Max Len: 14
  • Distro: <5: 0, 5-10